# vulnrag — smoke / eval

Детерминированный smoke-тест: подсевает **контрольную** уязвимость (`ctrl-lib`, уязвим в `[1.0, 2.0)`) в реальный Qdrant и проверяет все четыре вердикта независимо от того, что наунсилось бэкафиллом.

Требует поднятый **Qdrant** и **Ollama** (модели `bge-m3`, `qwen2.5:3b`). Подключается к РЕАЛЬНОМУ Qdrant через `build_components()` (не `:memory:`).

Случаи `vulnerable` и `unconfirmed` дёргают LLM — на CPU по ~30–60 с каждый.

In [ ]:
from datetime import date
from vulnrag.factory import build_components
from vulnrag.models import AffectedProduct, Vulnerability
from vulnrag.index.embedder import index_vulnerabilities
from vulnrag.query.answer import answer

store, emb, llm = build_components()   # реальный Qdrant + Ollama
store.ensure_collection()

# Контрольная уязвимость: ctrl-lib уязвим в [1.0, 2.0)
ctrl = Vulnerability(
    cve_id='CVE-CTRL-0001', title='control',
    description='Control test vulnerability in ctrl-lib.',
    severity='HIGH', cvss_score=7.5, fixed_versions=['2.0'],
    affected=[AffectedProduct(product='ctrl-lib', version_start='1.0',
              version_end='2.0', version_start_incl=True, version_end_incl=False)],
    published=date(2024, 1, 1), last_modified=date(2024, 1, 1))
index_vulnerabilities([ctrl], embedder=emb, store=store)

GOLDEN = [
    ('Is ctrl-lib 1.5 safe?', 'vulnerable'),       # версия в уязвимом диапазоне
    ('Is ctrl-lib 2.5 safe?', 'safe'),             # компонент есть, версия вне диапазона
    ('Is ctrl-lib vulnerable?', 'unconfirmed'),    # версия не указана
    ('Is totally-made-up-lib 1.0 safe?', 'not_found'),
]

ok = True
for q, expected in GOLDEN:
    r = answer(q, embedder=emb, store=store, llm=llm)
    passed = r.verdict == expected
    ok = ok and passed
    print(f"[{'OK ' if passed else 'FAIL'}] {q}  ->  {r.verdict}  (expected {expected})")
    print('       CVEs:', [c['cve_id'] for c in r.cves])

print('\nИтог:', 'ВСЕ ПРОШЛИ' if ok else 'ЕСТЬ ОТКЛОНЕНИЯ')
assert ok, 'golden-кейсы разошлись с ожиданиями'